<a href="https://colab.research.google.com/github/Daviabrt/Mercado-de-Energia/blob/main/PySDDP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
Aula 11

AaULA 1 mARCATO


In [1]:
!pip install PySDDP==0.0.18

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.6/78.6 kB 5.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.0/138.0 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 45.3 MB/s eta 0:00:00
  Created wheel for typing: filename=typing-3.7.4.3-py3-none-any.whl size=26304 sha256=67c6200815cf6674bee822fde5a3c413beb265fe8ea40f88cf486c2d9c2026a2
  Stored in directory: /root/.cache/pip/wheels/12/98/52/2bffe242a9a487f00886e43b8ed8dac46456702e11a0d6abef
Successfully built typing


Montar pasta arquivos de Entrada

In [11]:
from google.colab import drive
import os
from google.colab import files
drive.mount('/content/gdrive') #Montar o Google Drive
print(os.getcwd())
caminho = 'gdrive/My Drive/Newave/' # qual diretorio estou

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).
/content


In [12]:
from PySDDP.Pen import Newave

casoestudo = Newave(caminho)

OK! Leitura do CASO.DAT realizada com sucesso.
OK! Leitura do ARQUIVOS.DAT realizada com sucesso.
OK! Leitura do HIDR.DAT realizada com sucesso.
OK! Leitura do VAZOES.DAT realizada com sucesso.
OK! Leitura do CONFHD.DAT realizada com sucesso.


In [30]:
Usina = casoestudo.confhd.get('ItaIpu')
print(f"Nome {Usina['nome']}") # Printar o que eu quero da Usina, vol min etc....

Nome ITAIPU      


In [32]:
print(Usina.keys())

dict_keys(['codigo', 'nome', 'posto', 'ree', 'vol_ini', 'status', 'modif', 'ano_i', 'ano_f', 'bdh', 'sist', 'empr', 'jusante', 'desvio', 'vol_min', 'vol_max', 'vol_vert', 'vol_min_desv', 'cota_min', 'cota_max', 'pol_cota_vol', 'pol_cota_area', 'coef_evap', 'num_conj_maq', 'maq_por_conj', 'pef_por_conj', 'cf_hbqt', 'cf_hbqt_2', 'cf_hbqt_3', 'cf_hbqt_4', 'cf_hbqt_5', 'cf_hbqg', 'cf_hbqg_2', 'cf_hbqg_3', 'cf_hbqg_4', 'cf_hbqg_5', 'cf_hbpt', 'cf_hbpt_2', 'cf_hbpt_3', 'cf_hbpt_4', 'cf_hbpt_5', 'alt_efet_conj', 'vaz_efet_conj', 'prod_esp', 'perda_hid', 'num_pol_vnj', 'pol_vaz_niv_jus', 'pol_vaz_niv_jus_2', 'pol_vaz_niv_jus_3', 'pol_vaz_niv_jus_4', 'pol_vaz_niv_jus_5', 'cota_ref_nivel_jus', 'cfmed', 'inf_canal_fuga', 'fator_carga_max', 'fator_carga_min', 'vaz_min', 'unid_base', 'tipo_turb', 'repres_conj', 'teifh', 'ip', 'tipo_perda', 'data', 'observ', 'vol_ref', 'tipo_reg', 'vazoes', 'vol_mint', 'vol_maxt', 'vol_minp', 'vaz_mint', 'cfugat', 'vol_util', 'pot_efet', 'vaz_efet', 'status_vol_mort

In [33]:
from cvxopt import matrix, solvers
import numpy as np


def Series_Sinteticas(Usina, AnoAnalisado, N_meses, Mes_analisado,imprime = False):
    """
    Função que calcula a série sintética de vazões de uma usina
    """


    # Retira o ultimo e o primeiro ano e considera que a contagem começa em zero
    N_anos = len(Usina['vazoes']) - 3

    AnoAnalisado = AnoAnalisado - 2023 + N_anos + 2

    if Mes_analisado - N_meses >= 0:

        recorte = Usina['vazoes'][1:N_anos+1,(Mes_analisado-N_meses):Mes_analisado]

    else:

        recorteAnoAtual = Usina['vazoes'][1:N_anos+1,0:Mes_analisado]

        recorteAnoAnt = Usina['vazoes'][0:N_anos,(Mes_analisado-N_meses):]

        recorte = np.hstack((recorteAnoAnt,recorteAnoAtual))


    identidade = np.eye(N_anos)

    Aeq = np.concatenate((recorte, identidade), axis = 1)
    Aeq = Aeq.astype('float')
    Aeq = matrix(Aeq)

    Beq = Usina['vazoes'][1:N_anos+1,Mes_analisado]
    Beq = Beq.astype('float')
    Beq = matrix(Beq)
    q = matrix(np.zeros(N_anos+N_meses))
    P = 2*np.eye(N_anos+N_meses)
    for i in range(0,N_meses):
        P[i][i] = 0
    P = matrix(P)




    A = np.vstack((-1*np.eye(N_anos+N_meses), np.eye(N_anos+N_meses)))
    A = A.astype('float')
    A = matrix(A)
    B = 99999*np.ones(((N_anos+N_meses)*2,1))

    B = B.astype('float')
    B = matrix(B)


    solvers.options['show_progress'] = False
    abstol = 1e-9
    reltol = 1e-9
    sol = solvers.qp(P, q, A, B, Aeq, Beq, abstol=abstol, reltol=reltol)

    fob = sum(sol['x'][N_meses:])

    if Mes_analisado - N_meses >= 0:

        soma = 0

        for i, v in enumerate(Usina['vazoes'][AnoAnalisado,(Mes_analisado-N_meses):Mes_analisado]):

            soma += v*sol['x'][i]

    else:

        soma = 0

        for i, v in enumerate(Usina['vazoes'][AnoAnalisado,0:Mes_analisado]):

            soma += v*sol['x'][i+N_meses-Mes_analisado]

        for i, v in enumerate(Usina['vazoes'][AnoAnalisado-1,(Mes_analisado-N_meses):]):

            soma += v*sol['x'][i]

    if imprime:

        print(f'Vazão estimada para o mês {Mes_analisado}: {soma} hm^3')

    return [fob,soma,sol['x']]


In [35]:
from cvxopt import matrix, solvers
from matplotlib import pyplot as plt



# Ano a ser analisado
AnoAnalisado = 2026

# Numero de meses anteriores a serem considerados
N_meses = 3

# Retira o ultimo e o primeiro ano e considera que a contagem começa em zero
N_anos = len(Usina['vazoes']) - 3

# Numero do mes considerando janeiro zero
Mes_analisado = 7

# Excluir anos mais recentes caso queira
# Usina['vazoes'] = Usina['vazoes'][:-2]


print(f'A vazão estimada para o mês {Mes_analisado + 1}, considerando os {N_meses} últimos meses para o cálculo, é de: {Series_Sinteticas(Usina, AnoAnalisado, N_meses, Mes_analisado)[1]} m³/s')

resultado = []
fob = []

for i in range(0,12):

    resposta = Series_Sinteticas(Usina, AnoAnalisado, N_meses, i)
    resultado.append(resposta[1])


for i in range(1,12):

    resposta = Series_Sinteticas(Usina, AnoAnalisado, i, Mes_analisado)
    fob.append(resposta[0])



# Converte o ano para o indice da lista
AnoAnalisado = AnoAnalisado - 2023 + N_anos + 2



plt.figure()
plt.title("Vazão Sintética x Vazão Real")
plt.xlabel("Meses")
plt.ylabel("Vazão (m³/s)")
plt.plot(list(range(1,13)),resultado,marker="o",color="blue",label="Vazão Sintética")
plt.plot(list(range(1,13)),Usina['vazoes'][AnoAnalisado],marker="o",color="red",label="Vazão Real")
plt.legend()


plt.figure()
plt.title("Erro Total Mês de Abril")
plt.xlabel("Meses Anteriores Considerados")
plt.ylabel("Erro")
plt.plot(list(range(1,12)),fob,marker="o",color="blue")

IndexError: index 98 is out of bounds for axis 0 with size 96